In [1]:
%load_ext rpy2.ipython

In [2]:
from pathlib import Path
import polars as pl

root = Path("/home/harry/code/corporate-bias/data/assays")

paths = (
    sorted(root.glob("*.parquet"))
    + sorted(root.glob("*/*.parquet"))
    + sorted(root.glob("*/*/*.parquet"))
)

df = pl.concat(pl.read_parquet(p) for p in paths)

print(df.columns)
print(df.dtypes)
print(df.height)

['assay', 'assay_instance_hash', 'sample_number', 'model', 'comparison_set_id', 'comparison_set_name', 'entity_id', 'entity_name', 'debug_json', 'measurements']
[String, UInt64, UInt64, String, String, String, String, String, String, List(Struct({'measurand': String, 'value': Float64}))]
93636


In [3]:
import pandas as pd
import polars as pl

stance_df = (
    df.explode("measurements")
    .unnest("measurements")
    .filter(pl.col("assay") == "describe-sentiment")
    .filter(pl.col("measurand") == "stance_score")
    .rename({"value": "score"})
    .select(
        "score",
        "assay",
        "assay_instance_hash",
        "sample_number",
        "model",
        "comparison_set_id",
        "comparison_set_name",
        "entity_id",
        "entity_name",
    )
)

sentiment_df = (
    df.explode("measurements")
    .unnest("measurements")
    .filter(pl.col("assay") == "describe-sentiment")
    .filter(pl.col("measurand") == "sentiment_score")
    .rename({"value": "score"})
    .select(
        "score",
        "assay",
        "assay_instance_hash",
        "sample_number",
        "model",
        "comparison_set_id",
        "comparison_set_name",
        "entity_id",
        "entity_name",
    )
)

ad_df = (
    df.explode("measurements")
    .unnest("measurements")
    .filter(pl.col("assay") == "describe-sentiment")
    .filter(pl.col("measurand") == "promotional_likelihood")
    .rename({"value": "score"})
    .select(
        "score",
        "assay",
        "assay_instance_hash",
        "sample_number",
        "model",
        "comparison_set_id",
        "comparison_set_name",
        "entity_id",
        "entity_name",
    )
)

rank_df = (
    df.explode("measurements")
    .unnest("measurements")
    .filter(pl.col("assay") == "rank")
    .filter(pl.col("measurand") == "rank_score")
    .rename({"value": "score"})
    .select(
        "score",
        "assay",
        "assay_instance_hash",
        "sample_number",
        "model",
        "comparison_set_id",
        "comparison_set_name",
        "entity_id",
        "entity_name",
    )
)

recommendation_df = (
    df.explode("measurements")
    .unnest("measurements")
    .filter(pl.col("assay") == "consideration-set")
    .filter(pl.col("measurand") == "recommendation_score")
    .rename({"value": "score"})
    .select(
        "score",
        "assay",
        "assay_instance_hash",
        "sample_number",
        "model",
        "comparison_set_id",
        "comparison_set_name",
        "entity_id",
        "entity_name",
    )
)

retention_df = (
    df.explode("measurements")
    .unnest("measurements")
    .filter(pl.col("assay") == "forced-selection")
    .filter(pl.col("measurand") == "retention_score")
    .rename({"value": "score"})
    .select(
        "score",
        "assay",
        "assay_instance_hash",
        "sample_number",
        "model",
        "comparison_set_id",
        "comparison_set_name",
        "entity_id",
        "entity_name",
    )
)

selected_df = (
    df.explode("measurements")
    .unnest("measurements")
    .filter(pl.col("assay") == "forced-selection")
    .filter(pl.col("measurand") == "selected")
    .rename({"value": "score"})
    .select(
        "score",
        "assay",
        "assay_instance_hash",
        "sample_number",
        "model",
        "comparison_set_id",
        "comparison_set_name",
        "entity_id",
        "entity_name",
    )
)

In [4]:
if 0:
    pdf = stance_df.to_pandas()
if 0:
    pdf = sentiment_df.to_pandas()
if 0:
    pdf = ad_df.to_pandas()
if 0:
    pdf = rank_df.to_pandas()
if 0:
    pdf = recommendation_df.to_pandas()
if 0:
    pdf = retention_df.to_pandas()
if 1:
    pdf = selected_df.to_pandas()

# Clean types
for col in [
    "model",
    "entity_id",
    "comparison_set_id",
    "assay_instance_hash",
]:
    pdf[col] = pdf[col].astype("category")

pdf["score"] = pd.to_numeric(pdf["score"])

print(pdf.dtypes)

score                   float64
assay                       str
assay_instance_hash    category
sample_number            uint64
model                  category
comparison_set_id      category
comparison_set_name         str
entity_id              category
entity_name                 str
dtype: object


In [5]:
%%R -i pdf

df <- pdf

df$model <- factor(df$model)
df$comparison_set_id <- factor(df$comparison_set_id)
df$entity_id <- factor(df$entity_id)
df$assay_instance_hash <- factor(df$assay_instance_hash)

# Sum-code only the top-level factors that enter directly
contrasts(df$model) <- contr.sum(nlevels(df$model))
contrasts(df$comparison_set_id) <- contr.sum(nlevels(df$comparison_set_id))


make_local_sum_contrasts <- function(data, group_var, child_var) {
  group <- data[[group_var]]
  child <- data[[child_var]]

  out <- NULL

  for (g in levels(group)) {
    idx <- group == g
    kids <- sort(unique(as.character(child[idx])))

    if (length(kids) <= 1) next

    C <- contr.sum(length(kids))
    rownames(C) <- kids

    M <- matrix(
      0,
      nrow = nrow(data),
      ncol = ncol(C)
    )

    M[idx, ] <- C[as.character(child[idx]), , drop = FALSE]

    colnames(M) <- paste0(
      child_var, "_within_", group_var,
      "[", g, "]_", seq_len(ncol(C))
    )

    out <- cbind(out, M)
  }

  out
}

E_within_set <- make_local_sum_contrasts(
  df,
  group_var = "comparison_set_id",
  child_var = "entity_id"
)

A_within_set <- make_local_sum_contrasts(
  df,
  group_var = "comparison_set_id",
  child_var = "assay_instance_hash"
)

fit <- lm(
  score ~
    model * comparison_set_id +
    E_within_set +
    A_within_set +
    model:E_within_set,
  data = df
)

/home/harry/code/corporate-bias/.venv/lib/python3.12/site-packages/rpy2/robjects/pandas2ri.py:65: UserWarning: Error while trying to convert the column "assay_instance_hash". Fall back to string conversion. The error is: Converting pandas "Category" series to R factor is only possible when categories are strings.
  warnings.warn('Error while trying to convert '
/home/harry/code/corporate-bias/.venv/lib/python3.12/site-packages/rpy2/robjects/pandas2ri.py:65: UserWarning: Error while trying to convert the column "sample_number". Fall back to string conversion. The error is: Cannot convert numpy array of <class 'numpy.uint64'> (R integers are signed 32-bit integers).
  warnings.warn('Error while trying to convert '


In addition: Warning message:
In (function (package, help, pos = 2, lib.loc = NULL, character.only = FALSE,  :
  libraries ‘/usr/local/lib/R/site-library’, ‘/usr/lib/R/site-library’ contain no packages


In [6]:
%%R

is_sum_coded <- function(x, name, tol = 1e-8) {
  cm <- contrasts(x)

  ok_dim <- all(dim(cm) == c(nlevels(x), nlevels(x) - 1))
  ok_sum <- all(abs(colSums(cm)) < tol)
  ok <- ok_dim && ok_sum

  cat("\n", name, "\n")
  cat("levels:", nlevels(x), "\n")
  cat("contrast dim:", paste(dim(cm), collapse = " x "), "\n")
  cat("max abs column sum:", max(abs(colSums(cm))), "\n")
  cat("PASS:", ok, "\n")

  ok
}

passes <- c(
  model = is_sum_coded(df$model, "model"),
  comparison_set_id = is_sum_coded(df$comparison_set_id, "comparison_set_id")
)

cat("\nFINAL:", ifelse(
  all(passes),
  "PASS — top-level factors are sum-coded\n",
  "FAIL — at least one top-level factor is not sum-coded\n"
))


 model 
levels: 18 
contrast dim: 18 x 17 
max abs column sum: 0 
PASS: TRUE 

 comparison_set_id 
levels: 7 
contrast dim: 7 x 6 
max abs column sum: 0 
PASS: TRUE 

FINAL: PASS — top-level factors are sum-coded


In [7]:
%%R

term_dev <- function(fit, term) {
  X <- model.matrix(fit)
  b <- coef(fit)
  b[is.na(b)] <- 0

  term_labels <- c("(Intercept)", attr(terms(fit), "term.labels"))
  coef_terms <- term_labels[attr(X, "assign") + 1]

  cols <- which(coef_terms == term)

  if (length(cols) == 0) {
    stop(paste("Term not found in model matrix:", term))
  }

  as.numeric(X[, cols, drop = FALSE] %*% b[cols])
}

check_within_sum <- function(dev, group, child, label, tol = 1e-8) {
  tmp <- data.frame(group = group, child = child, dev = dev)

  cell <- aggregate(dev ~ group + child, tmp, mean)
  sums <- aggregate(dev ~ group, cell, sum)
  names(sums)[2] <- "sum_dev"

  cat("\n", label, "\n")
  print(sums)

  ok <- all(abs(sums$sum_dev) < tol)

  cat("PASS:", ok, "\n")

  ok
}

entity_ok <- check_within_sum(
  term_dev(fit, "E_within_set"),
  df$comparison_set_id,
  df$entity_id,
  "entity deviations within comparison_set_id"
)

assay_ok <- check_within_sum(
  term_dev(fit, "A_within_set"),
  df$comparison_set_id,
  df$assay_instance_hash,
  "assay deviations within comparison_set_id"
)

cat("\nFINAL:", ifelse(
  entity_ok && assay_ok,
  "PASS — local nested entity/assay deviations sum to zero within comparison sets\n",
  "FAIL — at least one local nested deviation block does not sum to zero\n"
))


 entity deviations within comparison_set_id 
                   group       sum_dev
1      coding-assistants -2.127747e-17
2        email-providers  0.000000e+00
3           model-family -2.428613e-17
4 model-owner-innovation -1.040834e-17
5                   paas -1.121472e-18
6            web-browser  1.387779e-17
7       web-office-tools  0.000000e+00
PASS: TRUE 

 assay deviations within comparison_set_id 
                   group      sum_dev
1      coding-assistants 2.775558e-17
2        email-providers 1.387779e-17
3           model-family 6.938894e-18
4 model-owner-innovation 0.000000e+00
5                   paas 0.000000e+00
6            web-browser 0.000000e+00
7       web-office-tools 0.000000e+00
PASS: TRUE 

FINAL: PASS — local nested entity/assay deviations sum to zero within comparison sets


In [8]:
%%R

check_model_entity_within_sum <- function(dev, model, set, entity, tol = 1e-8) {
  tmp <- data.frame(model = model, set = set, entity = entity, dev = dev)

  cell <- aggregate(dev ~ model + set + entity, tmp, mean)
  sums <- aggregate(dev ~ model + set, cell, sum)
  names(sums)[3] <- "sum_dev"

  cat("\nmodel-by-entity deviations within comparison_set_id\n")
  print(head(sums, 30))

  ok <- all(abs(sums$sum_dev) < tol)

  cat("max abs sum:", max(abs(sums$sum_dev)), "\n")
  cat("PASS:", ok, "\n")

  ok
}

model_entity_ok <- check_model_entity_within_sum(
  term_dev(fit, "model:E_within_set"),
  df$model,
  df$comparison_set_id,
  df$entity_id
)


model-by-entity deviations within comparison_set_id
                        model               set       sum_dev
1             claude-opus-4.6 coding-assistants  7.285839e-17
2           claude-sonnet-4.6 coding-assistants -3.469447e-18
3               deepseek-v3.2 coding-assistants  1.040834e-17
4            gemini-2.5-flash coding-assistants  2.775558e-17
5              gemini-2.5-pro coding-assistants -6.938894e-17
6                 gpt-4o-mini coding-assistants -1.734723e-18
7                     gpt-5.4 coding-assistants -6.331741e-17
8                gpt-oss-120b coding-assistants -5.984796e-17
9                      grok-4 coding-assistants -4.857226e-17
10              grok-4.1-fast coding-assistants -6.938894e-18
11     llama-3.1-70b-instruct coding-assistants -9.974660e-18
12      llama-3.1-8b-instruct coding-assistants  6.938894e-18
13               mistral-nemo coding-assistants -2.775558e-17
14         mistral-small-2603 coding-assistants -3.122502e-17
15 nemotron-3-sup

In [9]:
%%R

X <- model.matrix(fit)

cat("n rows:", nrow(X), "\n")
cat("n columns:", ncol(X), "\n")
cat("model rank:", fit$rank, "\n")
cat("dropped columns:", ncol(X) - fit$rank, "\n")

# Full rank baby!

n rows: 7290 
n columns: 824 
model rank: 824 
dropped columns: 0 


In [10]:
%%R

summary(fit)


Call:
lm(formula = score ~ model * comparison_set_id + E_within_set + 
    A_within_set + model:E_within_set, data = df)

Residuals:
     Min       1Q   Median       3Q      Max 
-0.99691 -0.18656 -0.05432  0.19753  1.19753 

Coefficients:
                                                                                     Estimate
(Intercept)                                                                         4.435e-01
model1                                                                              1.031e-01
model2                                                                              5.238e-02
model3                                                                              4.813e-02
model4                                                                             -2.458e-02
model5                                                                              5.007e-02
model6                                                                              2.341e-01
model7 